In [1]:
from configuracoes_notebooks import set_proj_dir
set_proj_dir()

O diretorio do seu projeto é coleta_cebrap
Caminho absoluto do diretorio encontrado C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap
Caminho no path.


In [2]:
import geopandas as gpd
import os

from notebooks.jupyter import utils
from utils import (
    get_data_diretorio,
    save_parquet_excel
)
from utils.downloads import download_malha_geosampa

# Área de Inundação no Território / Total de área do distrito

In [3]:
data_path= get_data_diretorio()
assets_path = os.path.join(
    data_path,
    'assets'
)

# Dependências

Este notebook é dependente dos parquets resultantes dos notebooks "overlay_mancha_inund_distrit" e "../../arborizacao_viaria/malha_distritos"

In [4]:
gdf_distrito = gpd.read_parquet(
    os.path.join(
        data_path,
        'assets',
        'distrito_ibge.parquet'
    )
)

In [5]:
gdf_overlay_inund = gpd.read_parquet(
    os.path.join(
        assets_path,
        'areas_inundacao',
        'overlay_mancha_inund_dist.parquet'
    )
)

# Soma das Áreas de Inundação por Distrito

In [6]:
gdf_overlay_inund.columns

Index(['cd_mancha_inund', 'nm_bacia_h', 'qt_area_me', 'qt_elevaca',
       'qt_cota_in', 'qt_profund', 'area_mancha_inund', 'CD_MUN', 'NM_MUN',
       'CD_DIST', 'NM_DIST', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI',
       'CD_CONCURB', 'NM_CONCURB', 'AREA_KM2', 'total_pop', 'total_dom',
       'geometry', 'area_inund_recort'],
      dtype='object')

Para calcular a porcentagem de área de inundação por Distrito, primeiro calculamos o total de área de inundação por distrito.

In [7]:
for distrito in gdf_distrito['CD_DIST']:
    gdf_distrito.loc[gdf_distrito['CD_DIST']==distrito, 'area_inund']=(
        sum(
            gdf_overlay_inund.loc[gdf_overlay_inund['CD_DIST']==distrito, 'area_inund_recort']
        )
    )

In [8]:
gdf_distrito.sample(3)

,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,area_inund
24,3550308,São Paulo,355030825,CIDADE TIRADENTES,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,14.912833,194177,76047,"POLYGON ((356929.803 7388433.519, 356929.413 7...",50407.931716
89,3550308,São Paulo,355030890,VILA MARIANA,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,8.523343,127286,70691,"POLYGON ((332625.627 7389136.896, 332589.427 7...",104482.495817
14,3550308,São Paulo,355030815,CAMPO BELO,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,8.842744,71034,36663,"POLYGON ((329439.691 7384123.427, 329399.519 7...",95326.617687


# Salvar GDF

In [9]:
save_parquet_excel(
    gdf_distrito,
    'area_inundacao_distrito',
    assets_path,
    data_subpath='areas_inundacao'
)